# RTX-KG2 Data Explorer
Explore the nodes and edges TSV files without loading them fully into memory.  
Uses **DuckDB** to query on-disk — safe for 50 GB+ files.


In [ ]:
# Install dependencies (run once)
# %pip install duckdb pandas


In [3]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# ── EDIT THESE PATHS to point at your actual files ──────────────────────────
NODES_FILE   = "rtx-kg2_nodes_2.8.1.tsv"
EDGES_FILE   = "rtx-kg2_edges_2.8.1.tsv"
# These KG2 TSVs are a Neo4j bulk-import export: the data files have NO header
# row. The column names live in these separate *_header_* files.
NODES_HEADER = "rtx-kg2_nodes_header_2.8.1.tsv"
EDGES_HEADER = "rtx-kg2_edges_header_2.8.1.tsv"
# ────────────────────────────────────────────────────────────────────────────

# Read column names from the header files. rstrip("\r\n") matters: these files
# are CRLF, so a bare rstrip("\n") would leave a stray "\r" on the LAST column
# name (e.g. "category\r"), silently breaking any reference to it.
with open(NODES_HEADER) as f:
    node_cols = f.readline().rstrip("\r\n").split("\t")
with open(EDGES_HEADER) as f:
    edge_cols = f.readline().rstrip("\r\n").split("\t")

# Reusable read_csv() sources. Every query below does `FROM {NODES_SRC}` etc.
# Key options:
#   header=false + names=...  -> the data files have no header row
#   max_line_size=...         -> some node rows exceed 3 MB (huge synonym lists),
#                                over DuckDB's 2 MB default; raise it or scans crash
MAX_LINE = 20_000_000
NODES_SRC = (f"read_csv('{NODES_FILE}', delim='\t', header=false, quote='', "
             f"all_varchar=true, max_line_size={MAX_LINE}, names={node_cols})")
EDGES_SRC = (f"read_csv('{EDGES_FILE}', delim='\t', header=false, quote='', "
             f"all_varchar=true, max_line_size={MAX_LINE}, names={edge_cols})")

# Open a DuckDB connection (in-memory, no database file needed)
con = duckdb.connect()

print("DuckDB ready.")
print(f"  Nodes: {NODES_FILE}  ({len(node_cols)} columns)")
print(f"  Edges: {EDGES_FILE}  ({len(edge_cols)} columns)")

DuckDB ready.
  Nodes: rtx-kg2_nodes_2.8.1.tsv  (16 columns)
  Edges: rtx-kg2_edges_2.8.1.tsv  (18 columns)


## Nodes — first 100 rows

In [ ]:
nodes_100 = con.execute(f"""
    SELECT *
    FROM {NODES_SRC}
    LIMIT 100
""").df()

print(f"Columns ({len(nodes_100.columns)}): {list(nodes_100.columns)}")
nodes_100

## Edges — first 100 rows

In [10]:
# Real edges — raw, straight from the edges file, past the junk.
#
# `LIMIT 100` on the raw file returns the FIRST 100 physical rows, which are
# all biolink meta-model definitions (vocabulary about the graph itself, not
# biology). Their endpoints are CURIEs like 'biolink:has_part'. Real biological
# edges point at entities like 'NCIT:C3992', 'CHEMBL:...', 'MONDO:...'.
# So we just exclude endpoints starting with 'biolink:' — no nodes JOIN, no
# name resolution. This is purely what the EDGES file looks like.
edges_real = con.execute(f"""
    SELECT *
    FROM {EDGES_SRC}
    WHERE ":START_ID" NOT LIKE 'biolink:%'
      AND ":END_ID"   NOT LIKE 'biolink:%'
      AND "core_predicate" NOT LIKE 'biolink:close%'
      AND "core_predicate" NOT LIKE 'biolink:subclass%'
      AND "core_predicate" NOT LIKE 'biolink:has%'
      AND "core_predicate" NOT LIKE 'biolink:coexists%'
      AND "core_predicate" NOT LIKE 'biolink:related%'
      AND "core_predicate" NOT LIKE 'biolink:derives%'
      AND "core_predicate" NOT LIKE 'biolink:precedes%'
      AND "core_predicate" NOT LIKE 'biolink:develops_f%'
    LIMIT 100
""").df()

print(f"Columns ({len(edges_real.columns)}): {list(edges_real.columns)}")
edges_real


Columns (18): ['core_predicate', 'id', 'knowledge_source:string[]', 'negated', ':END_ID', 'predicate_label', 'publications:string[]', 'publications_info', 'qualified_object_aspect', 'qualified_object_direction', 'qualified_predicate', 'relation_label', 'source_predicate', ':START_ID', 'update_date', 'predicate:TYPE', 'subject', 'object']


,core_predicate,id,knowledge_source:string[],negated,:END_ID,predicate_label,publications:string[],publications_info,qualified_object_aspect,qualified_object_direction,qualified_predicate,relation_label,source_predicate,:START_ID,update_date,predicate:TYPE,subject,object
0,biolink:located_in,FMA:74535---FMA:location_of---None---None---None---FMA:296704---umls_source:FMA,infores:fma-umls,False,FMA:74535,location_of,None,{},None,None,None,INVERTED:location_of,FMA:location_of,FMA:296704,2022,FMA:location_of,FMA:296704,FMA:74535
1,biolink:physically_interacts_with,FMA:9592---FMA:adheres_to---None---None---None---FMA:9591---umls_source:FMA,infores:fma-umls,False,FMA:9591,adheres_to,None,{},None,None,None,adheres_to,FMA:adheres_to,FMA:9592,2022,FMA:adheres_to,FMA:9592,FMA:9591
2,biolink:physically_interacts_with,FMA:83981---FMA:adheres_to---None---None---None---FMA:71236---umls_source:FMA,infores:fma-umls,False,FMA:71236,adheres_to,None,{},None,None,None,adheres_to,FMA:adheres_to,FMA:83981,2022,FMA:adheres_to,FMA:83981,FMA:71236
3,biolink:physically_interacts_with,FMA:83982---FMA:adheres_to---None---None---None---FMA:71237---umls_source:FMA,infores:fma-umls,False,FMA:71237,adheres_to,None,{},None,None,None,adheres_to,FMA:adheres_to,FMA:83982,2022,FMA:adheres_to,FMA:83982,FMA:71237
4,biolink:located_in,FMA:293873---FMA:has_location---None---None---None---FMA:74534---umls_source...,infores:fma-umls,False,FMA:74534,has_location,None,{},None,None,None,has_location,FMA:has_location,FMA:293873,2022,FMA:has_location,FMA:293873,FMA:74534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,biolink:regulates,GO:0008045---GO:regulated_by---None---None---None---GO:1905812---umls_source:GO,infores:go,False,GO:0008045,regulated_by,None,{},None,None,None,INVERTED:regulated_by,GO:regulated_by,GO:1905812,2021,GO:regulated_by,GO:1905812,GO:0008045
96,biolink:regulates,GO:2000527---GO:regulates---None---None---None---GO:0002408---umls_source:GO,infores:go,False,GO:0002408,regulates,None,{},None,None,None,regulates,GO:regulates,GO:2000527,2021,GO:regulates,GO:2000527,GO:0002408
97,biolink:regulates,GO:2000529---GO:positively_regulates---biolink:causes---None---increased---G...,infores:go,False,GO:0002408,positively_regulates,None,{},None,increased,biolink:causes,positively_regulates,GO:positively_regulates,GO:2000529,2021,GO:positively_regulates,GO:2000529,GO:0002408
98,biolink:regulates,GO:2000528---GO:negatively_regulates---biolink:causes---None---decreased---G...,infores:go,False,GO:0002408,negatively_regulates,None,{},None,decreased,biolink:causes,negatively_regulates,GO:negatively_regulates,GO:2000528,2021,GO:negatively_regulates,GO:2000528,GO:0002408


Edges real data

In [ ]:
# Real edges — raw, straight from the edges file, past the junk.
#
# `LIMIT 100` on the raw file returns the FIRST 100 physical rows, which are
# all biolink meta-model definitions (vocabulary about the graph itself, not
# biology). Their endpoints are CURIEs like 'biolink:has_part'. Real biological
# edges point at entities like 'NCIT:C3992', 'CHEMBL:...', 'MONDO:...'.
# So we just exclude endpoints starting with 'biolink:' — no nodes JOIN, no
# name resolution. This is purely what the EDGES file looks like.
edges_real = con.execute(f"""
    SELECT *
    FROM {EDGES_SRC}
    WHERE ":START_ID" NOT LIKE 'biolink:%'
      AND ":END_ID"   NOT LIKE 'biolink:%'
      
    LIMIT 100
""").df()

print(f"Columns ({len(edges_real.columns)}): {list(edges_real.columns)}")
edges_real


In [ ]:
# Real edges — past the junk, and filtered to MEANINGFUL ones.
#
# `LIMIT 100` on the raw file returns the FIRST 100 physical rows = biolink
# meta-model definitions (vocabulary about the graph, not biology). And edges
# store only CURIEs (NCIT:C3992), never names. So here we:
#   1. filter to a real predicate (biolink:treats = drug -> disease)
#   2. drop negated edges (negated='True' = the KG asserts the edge is FALSE)
#   3. JOIN both endpoints to the nodes file to resolve IDs -> names
#   4. constrain categories so we get Drug -> Disease, not "Device treats Mouse"
SUBJECT_TYPES = ("biolink:Drug", "biolink:SmallMolecule", "biolink:ChemicalEntity")
OBJECT_TYPES  = ("biolink:Disease", "biolink:DiseaseOrPhenotypicFeature")
PREDICATE     = "biolink:treats"

edges_real = con.execute(f"""
    SELECT
        n_src."id:ID"    AS subject_id,
        n_src.name       AS subject,
        n_src.category   AS subject_type,
        e.core_predicate AS predicate,
        n_tgt."id:ID"    AS object_id,
        n_tgt.name       AS object,
        n_tgt.category   AS object_type
    FROM {EDGES_SRC} e
    JOIN {NODES_SRC} n_src ON n_src."id:ID" = e.":START_ID"
    JOIN {NODES_SRC} n_tgt ON n_tgt."id:ID" = e.":END_ID"
    WHERE e.negated = 'False'
      AND e.core_predicate = '{PREDICATE}'
      AND n_src.category IN {SUBJECT_TYPES}
      AND n_tgt.category IN {OBJECT_TYPES}
    LIMIT 100
""").df()

print(f"Showing {len(edges_real)} clean '{PREDICATE}' edges (Drug → Disease)")
edges_real


## Node category breakdown
What kinds of entities are in the graph?

In [ ]:
node_categories = con.execute(f"""
    SELECT
        category,
        COUNT(*) AS count
    FROM {NODES_SRC}
    GROUP BY category
    ORDER BY count DESC
""").df()

node_categories

## Edge predicate breakdown
What kinds of relationships exist? (`predicate:TYPE` column)

In [ ]:
edge_predicates = con.execute(f"""
    SELECT
        "predicate:TYPE" AS predicate,
        COUNT(*)         AS count
    FROM {EDGES_SRC}
    GROUP BY predicate
    ORDER BY count DESC
""").df()

edge_predicates

## Knowledge sources in edges
Which databases contribute edges?

In [ ]:
edge_sources = con.execute(f"""
    SELECT
        "knowledge_source:string[]" AS knowledge_source,
        COUNT(*)                    AS count
    FROM {EDGES_SRC}
    GROUP BY knowledge_source
    ORDER BY count DESC
    LIMIT 30
""").df()

edge_sources

## Look up a node by name
Change `search_name` to any drug, disease, or gene you care about.  
Uses `ILIKE` so it's case-insensitive and supports `%` wildcards.


In [ ]:
search_name = "%metformin%"   # ← change this

result = con.execute(f"""
    SELECT
        "id:ID"          AS id,
        name,
        category,
        knowledge_source,
        description
    FROM {NODES_SRC}
    WHERE name ILIKE '{search_name}'
       OR "synonym:string[]" ILIKE '{search_name}'
    LIMIT 20
""").df()

result

## Edges for a specific node
Paste a node ID from the lookup above into `node_id`.  
Shows all outgoing AND incoming edges for that node.


In [ ]:
node_id = "CHEMBL:CHEMBL1431"   # ← metformin; change this

edges_for_node = con.execute(f"""
    SELECT
        ":START_ID"                  AS source,
        ":END_ID"                    AS target,
        "predicate:TYPE"             AS predicate,
        "knowledge_source:string[]"  AS knowledge_source,
        negated
    FROM {EDGES_SRC}
    WHERE ":START_ID" = '{node_id}'
       OR ":END_ID"   = '{node_id}'
    LIMIT 100
""").df()

print(f"Found {len(edges_for_node)} edges (capped at 100)")
edges_for_node

## 2-hop paths: Drug → Intermediate → Disease
Preview of the kind of paths used for repurposing.  
Adjust `LIMIT` if you want more rows — but be careful on 50 GB files without a filter.


In [ ]:
# First register the files as views so we can JOIN them
con.execute(f"""
    CREATE OR REPLACE VIEW nodes AS
    SELECT
        "id:ID"          AS id,
        name,
        category,
        knowledge_source
    FROM {NODES_SRC}
""")

con.execute(f"""
    CREATE OR REPLACE VIEW edges AS
    SELECT
        ":START_ID"                  AS source,
        ":END_ID"                    AS target,
        "predicate:TYPE"             AS predicate,
        negated,
        "knowledge_source:string[]"  AS knowledge_source
    FROM {EDGES_SRC}
    WHERE negated = 'False'
""")

paths = con.execute("""
    SELECT
        n_drug.name                    AS drug,
        e1.predicate                   AS drug_to_target,
        n_mid.name                     AS intermediate,
        n_mid.category                 AS intermediate_type,
        e2.predicate                   AS target_to_disease,
        n_dis.name                     AS disease
    FROM edges e1
    JOIN nodes n_drug ON n_drug.id = e1.source
    JOIN nodes n_mid  ON n_mid.id  = e1.target
    JOIN edges e2     ON e2.source = n_mid.id
    JOIN nodes n_dis  ON n_dis.id  = e2.target
    WHERE n_drug.category IN ('biolink:Drug', 'biolink:SmallMolecule')
      AND n_dis.category  IN ('biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature')
      AND n_mid.category  IN ('biolink:Gene', 'biolink:Protein', 'biolink:GeneOrGeneProduct')
      AND e1.predicate IN ('biolink:inhibits', 'biolink:activates', 'biolink:binds',
                           'biolink:affects', 'biolink:interacts_with')
      AND e2.predicate IN ('biolink:associated_with', 'biolink:gene_associated_with_condition',
                           'biolink:contributes_to', 'biolink:causes')
    LIMIT 100
""").df()

print(f"Sample 2-hop paths: {len(paths)}")
paths

## Save a filtered subset to disk
Export just drugs + diseases + their connecting edges to a smaller TSV.  
Useful for loading into NetworkX or preparing Soufflé fact files.


In [ ]:
OUTPUT_NODES = "kg2_subset_nodes.tsv"
OUTPUT_EDGES = "kg2_subset_edges.tsv"

KEEP_CATEGORIES = [
    "biolink:Drug", "biolink:SmallMolecule",
    "biolink:Disease", "biolink:DiseaseOrPhenotypicFeature",
    "biolink:Gene", "biolink:GeneOrGeneProduct", "biolink:Protein",
    "biolink:Pathway", "biolink:BiologicalProcess", "biolink:MolecularActivity",
]
cat_list = ", ".join(f"'{c}'" for c in KEEP_CATEGORIES)

con.execute(f"""
    COPY (
        SELECT *
        FROM nodes
        WHERE category IN ({cat_list})
    ) TO '{OUTPUT_NODES}' (DELIMITER '\t', HEADER true)
""")

con.execute(f"""
    COPY (
        SELECT e.*
        FROM edges e
        JOIN nodes n_src ON n_src.id = e.source
        JOIN nodes n_tgt ON n_tgt.id = e.target
        WHERE n_src.category IN ({cat_list})
          AND n_tgt.category IN ({cat_list})
          AND e.negated = 'False'
    ) TO '{OUTPUT_EDGES}' (DELIMITER '\t', HEADER true)
""")

print(f"Saved filtered nodes → {OUTPUT_NODES}")
print(f"Saved filtered edges → {OUTPUT_EDGES}")